# Notebook 01 — Load Dữ liệu & Câu hỏi Trắc nghiệm
**Phụ trách:** Thành viên 1 (Data Engineer)

**Mục tiêu:**
- Load và kiểm tra toàn bộ 14 file CSV
- Encode và gộp 14 bảng lại với nhau tạo siêu ma trận
- Trả lời 10 câu hỏi trắc nghiệm từ dữ liệu


In [231]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from utils import load_all_data, parse_dates, check_nulls, check_duplicates, set_seed
from pathlib import Path
import os
pd.set_option('display.max_columns', None)
set_seed(42)



[seed] Random seed set to 42


## 1. Load dữ liệu

In [187]:
dfs = load_all_data()
dfs = parse_dates(dfs)

# Shorthand
products    = dfs['products']
customers   = dfs['customers']
orders      = dfs['orders']
order_items = dfs['order_items']
payments    = dfs['payments']
shipments   = dfs['shipments']
returns     = dfs['returns']
reviews     = dfs['reviews']
geography   = dfs['geography']
promotions  = dfs['promotions']
sales       = dfs['sales']
inventory   = dfs['inventory']
web_traffic = dfs['web_traffic']

  Loaded products.csv: (2412, 8)
  Loaded customers.csv: (121930, 7)
  Loaded promotions.csv: (50, 10)
  Loaded geography.csv: (39948, 4)
  Loaded orders.csv: (646945, 8)


C:\Users\skvat\PycharmProjects\Lanh-It-s-Du-Lieu\src\utils.py:47: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs[key] = pd.read_csv(path)


  Loaded order_items.csv: (714669, 7)
  Loaded payments.csv: (646945, 4)
  Loaded shipments.csv: (566067, 4)
  Loaded returns.csv: (39939, 7)
  Loaded reviews.csv: (113551, 7)
  Loaded sales.csv: (3833, 3)
  Loaded inventory.csv: (60247, 17)
  Loaded web_traffic.csv: (3652, 7)
  Loaded sample_submission.csv: (548, 3)


## 2. Kiểm tra dữ liệu

In [188]:
print('=== NULL VALUES ===')
display(check_nulls(dfs))

print('\n=== DUPLICATES ===')
display(check_duplicates(dfs))

=== NULL VALUES ===


,table,column,null_count,null_pct
2,order_items,promo_id_2,714463,99.97
0,promotions,applicable_category,40,80.00
1,order_items,promo_id,438353,61.34



=== DUPLICATES ===


,table,rows,duplicates
0,products,2412,0
1,customers,121930,0
2,promotions,50,0
3,geography,39948,0
4,orders,646945,0
5,order_items,714669,0
6,payments,646945,0
7,shipments,566067,0
8,returns,39939,0
9,reviews,113551,0


Các bảng đều không có duplicates promo_id 1, 2 và applicable_category đều được để null có chủ đích
Data được coi là đã làm sạch

## 3. Chỉnh sửa và encode dữ liệu
### 3.1. Chỉnh sửa master
Encode giá trị từ bảng master chuẩn bị cho phần 2 và 3
#### 3.1.1 encode products

Dựa vào miêu tả của bảng.
Ordinal encode size với map:

In [189]:
size_map = {'S': 1, 'M': 2, 'L': 3, 'XL': 4, 'XXL': 5}

In [190]:
products['size'] = products['size'].map(size_map).fillna(0)

xem cardinality của 3 cột: category, segment, color

In [191]:
cols_to_check = ['category', 'segment', 'color']

for col in cols_to_check:
    # Tính số lượng giá trị duy nhất (không tính Null)
    n_unique = products[col].nunique()
    # Lấy danh sách các giá trị đó
    unique_vals = products[col].dropna().unique()
    #in giá trị
    print(col)
    print(list(unique_vals))

category
['Streetwear', 'Casual', 'Outdoor', 'GenZ']
segment
['Everyday', 'Performance', 'Balanced', 'Standard', 'All-weather', 'Premium', 'Trendy', 'Activewear']
color
['green', 'silver', 'pink', 'yellow', 'red', 'black', 'orange', 'blue', 'white', 'purple']


cardinality của từng cột thấp. Chọn one-hot encode.
One-hot encode 3 cột và map giá trị từng cột.

In [192]:
cols_to_encode = ['category', 'segment', 'color']
products = pd.get_dummies(products, columns=cols_to_encode)


In [193]:
products.head()

,product_id,product_name,size,price,cogs,category_Casual,category_GenZ,category_Outdoor,category_Streetwear,segment_Activewear,segment_All-weather,segment_Balanced,segment_Everyday,segment_Performance,segment_Premium,segment_Standard,segment_Trendy,color_black,color_blue,color_green,color_orange,color_pink,color_purple,color_red,color_silver,color_white,color_yellow
0,536,SaigonFlex UC-01,1,11059.650000,9704.842875,False,False,False,True,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False
1,537,SaigonFlex UC-02,2,9523.076013,5393.870254,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,538,SaigonFlex UC-03,3,15951.633158,11371.919278,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,False,False
3,539,SaigonFlex UC-04,4,15753.717299,8573.172954,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True
4,540,SaigonFlex UC-05,1,15766.334536,14063.570406,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False


In [194]:
products.shape

(2412, 27)

#### 3.1.2 encode promotions

In [195]:
promotions.head()

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,NaN,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,NaN,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,NaN,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,NaN,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


Applicable_category miêu tả null là toàn bộ sản phẩm.
Chuyển thành 'all_categories' để chuẩn bị cho các phần tiếp theo

In [196]:
promotions['applicable_category'] = promotions['applicable_category'].fillna('all_categories')
promotions.head()

,promo_id,promo_name,promo_type,discount_value,start_date,end_date,applicable_category,promo_channel,stackable_flag,min_order_value
0,PROMO-0001,Spring Sale 2013,percentage,12.0,2013-03-18,2013-04-17,all_categories,email,1,0
1,PROMO-0002,Mid-Year Sale 2013,percentage,18.0,2013-06-23,2013-07-22,all_categories,online,0,0
2,PROMO-0003,Fall Launch 2013,percentage,10.0,2013-08-30,2013-10-02,all_categories,email,0,0
3,PROMO-0004,Year-End Sale 2013,percentage,20.0,2013-11-18,2014-01-02,all_categories,all_channels,0,50000
4,PROMO-0005,Urban Blowout 2013,fixed,50.0,2013-07-30,2013-09-02,Streetwear,online,0,150000


Xem cardinality cho promo_type và promo_channel

In [197]:
cols_to_check = ['promo_type', 'promo_channel']

for col in cols_to_check:
    # Tính số lượng giá trị duy nhất (không tính Null)
    n_unique = promotions[col].nunique()
    # Lấy danh sách các giá trị đó
    unique_vals = promotions[col].dropna().unique()
    #in giá trị
    print(col)
    print(list(unique_vals))

promo_type
['percentage', 'fixed']
promo_channel
['email', 'online', 'all_channels', 'in_store', 'social_media']


One-hot encode promo_type và promo_channel

In [198]:
cols_to_encode = ['promo_type', 'promo_channel']
promotions = pd.get_dummies(promotions, columns=cols_to_encode)

In [199]:
promotions.head()

,promo_id,promo_name,discount_value,start_date,end_date,applicable_category,stackable_flag,min_order_value,promo_type_fixed,promo_type_percentage,promo_channel_all_channels,promo_channel_email,promo_channel_in_store,promo_channel_online,promo_channel_social_media
0,PROMO-0001,Spring Sale 2013,12.0,2013-03-18,2013-04-17,all_categories,1,0,False,True,False,True,False,False,False
1,PROMO-0002,Mid-Year Sale 2013,18.0,2013-06-23,2013-07-22,all_categories,0,0,False,True,False,False,False,True,False
2,PROMO-0003,Fall Launch 2013,10.0,2013-08-30,2013-10-02,all_categories,0,0,False,True,False,True,False,False,False
3,PROMO-0004,Year-End Sale 2013,20.0,2013-11-18,2014-01-02,all_categories,0,50000,False,True,True,False,False,False,False
4,PROMO-0005,Urban Blowout 2013,50.0,2013-07-30,2013-09-02,Streetwear,0,150000,True,False,False,False,False,True,False


In [200]:
promotions.shape

(50, 15)

#### 3.1.3 encode customers

In [201]:
customers.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


Cardinality của các cột

In [202]:
cols_to_check = ['city', 'gender', 'age_group', 'acquisition_channel' ]

for col in cols_to_check:
    # Tính số lượng giá trị duy nhất (không tính Null)
    n_unique = customers[col].nunique()
    # Lấy danh sách các giá trị đó
    unique_vals = customers[col].dropna().unique()
    #in giá trị
    print(col)
    print(list(unique_vals))

city
['Hai Phong', 'Phu Ly', 'Viet Tri', 'Bac Giang', 'Lao Cai', 'Ha Long', 'Cam Pha', 'Son Tay', 'Ninh Binh', 'Bac Ninh', 'Nam Dinh', 'Hanoi', 'Uong Bi', 'Thai Nguyen', 'Hoi An', 'Kon Tum', 'Phan Rang-Thap Cham', 'Dong Hoi', 'Quang Ngai', 'Tuy Hoa', 'Hue', 'Da Nang', 'Phan Thiet', 'Quy Nhon', 'Nha Trang', 'Tam Ky', 'My Tho', 'Bien Hoa', 'Buon Ma Thuot', 'Bac Lieu', 'Tra Vinh', 'Vinh Long', 'Rach Gia', 'Long Xuyen', 'Vung Tau', 'Soc Trang', 'Pleiku', 'Can Tho', 'Ca Mau', 'Da Lat', 'Ben Tre', 'Ho Chi Minh City']
gender
['Female', 'Male', 'Non-binary']
age_group
['35-44', '45-54', '18-24', '55+', '25-34']
acquisition_channel
['social_media', 'email_campaign', 'organic_search', 'referral', 'direct', 'paid_search']


In [203]:
city_stats = customers['city'].value_counts(dropna=False)
display(city_stats)

city
Cam Pha                4398
Thai Nguyen            4347
Phu Ly                 4243
Hanoi                  4240
Ha Long                4236
Bac Ninh               4172
Hai Phong              4170
Nam Dinh               4169
Bac Giang              4160
Ninh Binh              4081
Son Tay                4075
Viet Tri               4054
Uong Bi                4026
Dong Hoi               3912
Kon Tum                3838
Lao Cai                3807
Hoi An                 3760
Phan Thiet             3749
Phan Rang-Thap Cham    3734
Hue                    3719
Tuy Hoa                3674
Quang Ngai             3616
Da Nang                3616
Tam Ky                 3562
Quy Nhon               3556
Nha Trang              3550
Ho Chi Minh City       1359
Vinh Long              1336
Pleiku                 1253
Vung Tau               1251
Da Lat                 1243
Can Tho                1214
Bac Lieu               1211
Ca Mau                 1211
Soc Trang              1208
My Tho         

Không thể frequency encode do một tỉnh có số lượng bằng chuyển sang rank encode.
one-hot encode tuổi và kênh.

rank encode cách tỉnh

In [204]:
city_counts = customers['city'].value_counts(dropna=False)
city_rank_df = city_counts.reset_index()
city_rank_df.columns = ['city', 'count']
city_rank_df['rank'] = city_rank_df['count'].rank(method='first', ascending=True).astype(int)
city_rank_map = dict(zip(city_rank_df['city'], city_rank_df['rank']))
customers['city'] = customers['city'].map(city_rank_map)

ordinal encode age_group

In [205]:
age_map = {'Unknown': 0, '18-24': 1, '25-34': 2, '35-44': 3, '45-54': 4, '55+': 5}
customers['age_group'] = customers['age_group'].map(age_map).fillna(0).astype(int)

one-hot encode gender, acquisition_channel

In [206]:
customers = pd.get_dummies(customers, columns=['gender', 'acquisition_channel'])

In [207]:
customers.head()

,customer_id,zip,city,signup_date,age_group,gender_Female,gender_Male,gender_Non-binary,acquisition_channel_direct,acquisition_channel_email_campaign,acquisition_channel_organic_search,acquisition_channel_paid_search,acquisition_channel_referral,acquisition_channel_social_media
0,1,15201,36,2021-12-30,3,True,False,False,False,False,False,False,False,True
1,2,15201,36,2013-12-27,4,True,False,False,False,True,False,False,False,False
2,3,15201,36,2018-07-24,1,True,False,False,False,False,True,False,False,False
3,4,15201,36,2017-11-29,3,False,True,False,False,False,False,False,True,False
4,5,15201,36,2022-09-23,5,False,True,False,False,False,True,False,False,False


In [208]:
customers.shape

(121930, 14)

#### 3.1.4 encode geography

In [209]:
geography.head()

,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13
3,15204,Bac Giang,East,District #13
4,15205,Bac Giang,East,District #13


cardinality của các cột

In [210]:
cols_to_check = ['region', 'district']

for col in cols_to_check:
    # Tính số lượng giá trị duy nhất (không tính Null)
    n_unique = geography[col].nunique()
    # Lấy danh sách các giá trị đó
    unique_vals = geography[col].dropna().unique()
    #in giá trị
    print(col)
    print(list(unique_vals))

region
['East', 'Central', 'West']
district
['District #13', 'District #04', 'District #06', 'District #05', 'District #03', 'District #01', 'District #14', 'District #15', 'District #02', 'District #10', 'District #12', 'District #09', 'District #16', 'District #18', 'District #17', 'District #08', 'District #11', 'District #07', 'District #19', 'District #25', 'District #30', 'District #32', 'District #21', 'District #24', 'District #28', 'District #20', 'District #29', 'District #27', 'District #22', 'District #31', 'District #26', 'District #23', 'District #36', 'District #35', 'District #38', 'District #39', 'District #37', 'District #34', 'District #33']


In [211]:
district_stats = geography['district'].value_counts(dropna=False)
display(district_stats)

district
District #25    1597
District #30    1518
District #19    1513
District #24    1511
District #07    1492
District #32    1459
District #21    1459
District #01    1337
District #33    1326
District #23    1210
District #03    1148
District #39    1113
District #15    1095
District #13    1083
District #29    1059
District #22    1056
District #09    1044
District #11    1033
District #02    1010
District #18     986
District #34     983
District #28     966
District #16     963
District #04     908
District #35     850
District #10     840
District #17     837
District #05     797
District #37     796
District #36     779
District #27     771
District #08     765
District #06     749
District #20     725
District #12     691
District #38     660
District #26     644
District #14     638
District #31     537
Name: count, dtype: int64

one hot encode region

one-hot encode region

In [212]:
geography = pd.get_dummies(geography, columns=['region'])

In [213]:
geography.head()

,zip,city,district,region_Central,region_East,region_West
0,15201,Hai Phong,District #13,False,True,False
1,15202,Phu Ly,District #13,False,True,False
2,15203,Viet Tri,District #13,False,True,False
3,15204,Bac Giang,District #13,False,True,False
4,15205,Bac Giang,District #13,False,True,False


### 3.2. liên kết các bảng trong transaction/sanity check.

In [214]:
# 1. Khởi tạo Base Table: Lấy order_items làm gốc
# Join với orders để lấy ngày tháng và thông tin khách hàng
print(f"[1] Base order_items: {len(order_items)} dòng.")
df_tx = pd.merge(order_items, orders, on='order_id', how='left')
print(f"    -> Sau khi join orders: {len(df_tx)} dòng.")

# 2. Join với payments (Quan hệ 1:1)
# LƯU Ý:
# - Bỏ 'payment_method' vì bảng orders đã có rồi.
# - Bỏ 'payment_value' để tránh lỗi nhân đôi doanh thu khi group by.
# Chỉ lấy 'installments' (Số kỳ trả góp)
pay_cols = ['order_id', 'installments']
df_tx = pd.merge(df_tx, payments[pay_cols], on='order_id', how='left')
print(f"[2] Sau khi join payments: {len(df_tx)} dòng.")

# 3. Join với shipments (Quan hệ 1:0/1)
ship_cols = ['order_id', 'ship_date', 'delivery_date', 'shipping_fee']
df_tx = pd.merge(df_tx, shipments[ship_cols], on='order_id', how='left')
print(f"[3] Sau khi join shipments: {len(df_tx)} dòng.")

# 4. Tiền xử lý và Join với returns
print("[4.1] Đang gom nhóm bảng returns để chống nhân bản...")
# Cộng dồn số lượng trả (sum), lấy ngày trả gần nhất (max), và gộp lý do (tùy chọn)
returns_agg = returns.groupby(['order_id', 'product_id']).agg({
    'return_quantity': 'sum',
    'return_date': 'max',
    'return_reason': lambda x: ', '.join(x.dropna().unique())
}).reset_index()

df_tx = pd.merge(df_tx, returns_agg, on=['order_id', 'product_id'], how='left')
print(f"[4.2] Sau khi join returns: {len(df_tx)} dòng.")

# 5. Tiền xử lý và Join với reviews
print("[5.1] Đang gom nhóm bảng reviews để chống nhân bản...")
# Nếu có nhiều review, lấy điểm rating trung bình (mean) hoặc mới nhất
reviews_agg = reviews.groupby(['order_id', 'product_id']).agg({
    'rating': 'mean',
    'review_date': 'max'
}).reset_index()

# Ép kiểu rating về int lại nếu cần
reviews_agg['rating'] = reviews_agg['rating'].round().astype('Int64')

df_tx = pd.merge(df_tx, reviews_agg, on=['order_id', 'product_id'], how='left')
print(f"[5.2] Sau khi join reviews: {len(df_tx)} dòng. (Hoàn tất!)")

# LÀM SẠCH NHANH SAU KHI JOIN
# Những đơn hàng không bị trả lại thì 'return_quantity' sẽ là NaN -> Điền 0
df_tx['return_quantity'] = df_tx['return_quantity'].fillna(0).astype(int)

# Đánh dấu cờ 'is_returned' (1: Có trả lại, 0: Không)
df_tx['is_returned'] = (df_tx['return_quantity'] > 0).astype('int8')

# Những đơn không có rating thì điền -1 (để phân biệt với rating 1-5 sao)
df_tx['rating'] = df_tx['rating'].fillna(-1).astype('int8')

[1] Base order_items: 714669 dòng.
    -> Sau khi join orders: 714669 dòng.
[2] Sau khi join payments: 714669 dòng.
[3] Sau khi join shipments: 714669 dòng.
[4.1] Đang gom nhóm bảng returns để chống nhân bản...
[4.2] Sau khi join returns: 714669 dòng.
[5.1] Đang gom nhóm bảng reviews để chống nhân bản...
[5.2] Sau khi join reviews: 714669 dòng. (Hoàn tất!)


Liên kết các bảng thành công

In [215]:
df_tx

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,installments,ship_date,delivery_date,shipping_fee,return_quantity,return_date,return_reason,rating,review_date,is_returned
0,1,2400,7,1138.22,0.0,NaN,NaN,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,3,2012-07-07,2012-07-11,1.37,0,NaT,NaN,5,2012-07-24,0
1,2,609,7,10166.25,0.0,NaN,NaN,2012-07-04,58621,1330,returned,cod,mobile,paid_search,1,2012-07-06,2012-07-10,2.60,6,2012-07-25,late_delivery,-1,NaT,1
2,3,396,3,11220.33,0.0,NaN,NaN,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,3,2012-07-04,2012-07-07,2.38,0,NaT,NaN,5,2012-08-03,0
3,4,635,5,10639.25,0.0,NaN,NaN,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,3,2012-07-05,2012-07-11,2.49,0,NaT,NaN,-1,NaT,0
4,6,1935,1,1597.84,0.0,NaN,NaN,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,1,2012-07-09,2012-07-16,25.79,0,NaT,NaN,-1,NaT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714664,834372,690,8,4473.92,0.0,NaN,NaN,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign,3,NaT,NaT,NaN,0,NaT,NaN,-1,NaT,0
714665,834377,1995,7,5250.79,0.0,NaN,NaN,2022-12-31,73046,37091,delivered,credit_card,mobile,referral,3,NaT,NaT,NaN,0,NaT,NaN,-1,NaT,0
714666,834387,2331,8,7389.06,0.0,NaN,NaN,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign,1,NaT,NaT,NaN,0,NaT,NaN,-1,NaT,0
714667,834392,1115,5,4767.33,0.0,NaN,NaN,2022-12-31,139431,93510,delivered,paypal,desktop,direct,1,NaT,NaT,NaN,0,NaT,NaN,-1,NaT,0


### 3.3. Tính doanh thu thực tế

In [216]:

df = df_tx.copy()
df_promotions = promotions
# Nếu đơn nào không dùng mã, 'promo_id' sẽ là NaN
df = pd.merge(df, df_promotions, on='promo_id', how='left')
print(f"-> Đã join bảng promotions. Số dòng: {len(df)}.")

# 1. TÍNH DOANH THU GỐC (Trạng thái chưa giảm giá)
df['base_revenue'] = df['unit_price'] * df['quantity']

# 2. TÍNH TỔNG GIÁ TRỊ ĐƠN HÀNG GỐC
# Dùng hàm transform('sum') để tính tổng theo order_id và gán lại cho từng dòng
df['order_base_total'] = df.groupby('order_id')['base_revenue'].transform('sum')

# 3. LÀM SẠCH DỮ LIỆU SAU KHI JOIN
# Những đơn không có mã KM (NaN) sẽ được điền bằng 0
cols_to_fill_0 = ['discount_value', 'min_order_value', 'promo_type_percentage', 'promo_type_fixed']
for col in cols_to_fill_0:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# 4. TÍNH TOÁN GIẢM GIÁ
# A. Tiền giảm theo %: (Doanh thu gốc * % giảm) * Cờ Percentage (1 hoặc 0)
discount_pct = df['base_revenue'] * (df['discount_value'] / 100.0) * df['promo_type_percentage']
# B. Tiền giảm cố định: Mức giảm * Cờ Fixed (1 hoặc 0)
# Giả định mức giảm fixed được áp dụng trực tiếp cho từng item
discount_fixed = df['discount_value'] * df['promo_type_fixed']

# Tổng giảm giá nháp
raw_discount = discount_pct + discount_fixed

# 5. ÁP DỤNG ĐIỀU KIỆN KINH DOANH
# Điều kiện 1: Đơn hàng phải lớn hơn hoặc bằng min_order_value
is_min_value_met = df['order_base_total'] >= df['min_order_value']

# Dùng np.where (như hàm IF trong Excel nhưng siêu nhanh)
valid_discount = np.where(is_min_value_met, raw_discount, 0.0)

# Điều kiện 2: Giảm giá không được làm doanh thu âm (Tối đa chỉ được giảm 100%)
# Dùng np.minimum để lấy mức trần là base_revenue
df['final_discount'] = np.minimum(valid_discount, df['base_revenue'])

# 6. CHỐT SỔ DOANH THU THỰC TẾ
df['actual_revenue'] = df['base_revenue'] - df['final_discount']

print("Tính toán hoàn tất! Đã tạo các cột 'actual_revenue' và 'gross_profit'.")

# Xóa các cột tạm thời
cols_to_drop = ['order_base_total', 'promo_type_percentage', 'promo_type_fixed', 'min_order_value', 'discount_value']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
transaction_final = df

-> Đã join bảng promotions. Số dòng: 714669.
-> Đang áp dụng công thức Vectorized cho 714,000+ dòng...
✅ Tính toán hoàn tất! Đã tạo các cột 'actual_revenue' và 'gross_profit'.


In [217]:
transaction_final

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,installments,ship_date,delivery_date,shipping_fee,return_quantity,return_date,return_reason,rating,review_date,is_returned,promo_name,start_date,end_date,applicable_category,stackable_flag,promo_channel_all_channels,promo_channel_email,promo_channel_in_store,promo_channel_online,promo_channel_social_media,base_revenue,final_discount,actual_revenue
0,1,2400,7,1138.22,0.0,NaN,NaN,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,3,2012-07-07,2012-07-11,1.37,0,NaT,NaN,5,2012-07-24,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7967.54,0.0,7967.54
1,2,609,7,10166.25,0.0,NaN,NaN,2012-07-04,58621,1330,returned,cod,mobile,paid_search,1,2012-07-06,2012-07-10,2.60,6,2012-07-25,late_delivery,-1,NaT,1,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71163.75,0.0,71163.75
2,3,396,3,11220.33,0.0,NaN,NaN,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,3,2012-07-04,2012-07-07,2.38,0,NaT,NaN,5,2012-08-03,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33660.99,0.0,33660.99
3,4,635,5,10639.25,0.0,NaN,NaN,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,3,2012-07-05,2012-07-11,2.49,0,NaT,NaN,-1,NaT,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53196.25,0.0,53196.25
4,6,1935,1,1597.84,0.0,NaN,NaN,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,1,2012-07-09,2012-07-16,25.79,0,NaT,NaN,-1,NaT,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1597.84,0.0,1597.84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714664,834372,690,8,4473.92,0.0,NaN,NaN,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign,3,NaT,NaT,NaN,0,NaT,NaN,-1,NaT,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35791.36,0.0,35791.36
714665,834377,1995,7,5250.79,0.0,NaN,NaN,2022-12-31,73046,37091,delivered,credit_card,mobile,referral,3,NaT,NaT,NaN,0,NaT,NaN,-1,NaT,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36755.53,0.0,36755.53
714666,834387,2331,8,7389.06,0.0,NaN,NaN,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign,1,NaT,NaT,NaN,0,NaT,NaN,-1,NaT,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59112.48,0.0,59112.48
714667,834392,1115,5,4767.33,0.0,NaN,NaN,2022-12-31,139431,93510,delivered,paypal,desktop,direct,1,NaT,NaT,NaN,0,NaT,NaN,-1,NaT,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23836.65,0.0,23836.65


### 3.4. biến đổi thời gian và gộp toàn bộ các bảng

In [219]:
# 1. Tạo bản sao để an toàn thao tác
df_final = transaction_final.copy()

# 2. Chuẩn bị biến thời gian (Date)
# Ép kiểu và chỉ lấy Ngày (bỏ Giờ Phút Giây để groupby chính xác)
df_final['Date'] = pd.to_datetime(df_final['order_date']).dt.date
# BƯỚC 1: Ráp nối Master Data
df_final = pd.merge(df_final, products, on='product_id', how='left')
df_final = pd.merge(df_final, customers, on='customer_id', how='left')
print(f"-> Đã ráp nối xong Master Data. Kích thước hiện tại: {df_final.shape}")
data_frame_final = df_final.copy()

# BƯỚC 2: TÍNH TOÁN LỢI NHUẬN GỘP (Đã có cogs từ bảng products)
df_final['total_cogs'] = df_final['cogs'] * df_final['quantity']
df_final['gross_profit'] = df_final['actual_revenue'] - df_final['total_cogs']

# BƯỚC 3: TẠO SIÊU ĐẶC TRƯNG TỪ ONE-HOT ENCODING (Vectorized)

# Tự động quét tìm các cột đã One-Hot trong bảng products
product_ohe_cols = [c for c in products.columns if c.startswith(('category_', 'segment_', 'color_'))]
advanced_revenue_cols = []

for col in product_ohe_cols:
    new_col_name = f"revenue_{col}"
    # Nhân Doanh thu với 1 (nếu thuộc nhãn đó) hoặc 0 (nếu không)
    df_final[new_col_name] = df_final['actual_revenue'] * df_final[col]
    advanced_revenue_cols.append(new_col_name)

# BƯỚC 4: GOM NHÓM THEO NGÀY (DAILY AGGREGATION)
# Định nghĩa các phép toán gom nhóm cơ bản
agg_funcs = {
    'actual_revenue': 'sum',           # Target Feature: Tổng doanh thu
    'gross_profit': 'sum',             # Tổng lợi nhuận gộp
    'order_id': 'nunique',             # Tổng số đơn hàng (Daily Orders)
    'quantity': 'sum',                 # Tổng số lượng sản phẩm bán ra
    'customer_id': 'nunique',          # Lượng khách active (Daily Active Users)
    'is_returned': 'sum'               # Tổng số sản phẩm bị trả lại
}

# Nạp thêm các phép tính Tổng cho toàn bộ các cột Advanced Revenue
for col in advanced_revenue_cols:
    agg_funcs[col] = 'sum'

# Thực thi Groupby
daily_timeseries = df_final.groupby('Date').agg(agg_funcs).reset_index()

# Sắp xếp lại theo thời gian (Bắt buộc với Time-Series để tránh Look-ahead bias)
daily_timeseries = daily_timeseries.sort_values('Date').reset_index(drop=True)

# Đổi tên cột Date thành lowercase cho chuẩn convention (tùy chọn)
daily_timeseries = daily_timeseries.rename(columns={'Date': 'date'})

print('Hoàn tất Giai đoạn 4")
# Hiển thị 5 dòng đầu tiên trên Notebook
display(daily_timeseries.head())

-> Đã ráp nối xong Master Data. Kích thước hiện tại: (714669, 77)
-> Đang tạo đặc trưng doanh thu theo nhãn (Category, Segment, Color)...
-> Đang Groupby theo ngày để tạo bảng Time-Series...
✅ Hoàn tất Giai đoạn 4! Bảng dữ liệu đã sẵn sàng.


,date,actual_revenue,gross_profit,order_id,quantity,customer_id,is_returned,revenue_category_Casual,revenue_category_GenZ,revenue_category_Outdoor,revenue_category_Streetwear,revenue_segment_Activewear,revenue_segment_All-weather,revenue_segment_Balanced,revenue_segment_Everyday,revenue_segment_Performance,revenue_segment_Premium,revenue_segment_Standard,revenue_segment_Trendy,revenue_color_black,revenue_color_blue,revenue_color_green,revenue_color_orange,revenue_color_pink,revenue_color_purple,revenue_color_red,revenue_color_silver,revenue_color_white,revenue_color_yellow
0,2012-07-04,5123547.94,1140556.750562,162,777,161,12,116436.2,47319.06,358947.83,4600844.85,340053.43,116436.2,1572352.0,2501959.77,426772.22,18894.4,99760.86,47319.06,689994.72,528310.93,928616.99,113127.03,168222.58,948753.89,494747.91,252465.3,521426.37,477882.22
1,2012-07-05,2751773.45,601193.216242,97,428,97,7,38829.92,59801.12,200817.52,2452324.89,160828.54,38829.92,818865.78,1487916.57,140468.39,39988.98,5074.15,59801.12,414333.97,167841.52,377319.17,101424.34,55456.23,619748.37,150204.6,191719.14,291988.76,381737.35
2,2012-07-06,3054029.42,536396.576639,93,441,93,8,86471.16,28441.71,163769.08,2775347.47,163769.08,86471.16,932551.0,1326982.31,476556.72,0.0,39257.44,28441.71,248983.78,135675.57,717565.24,211550.91,217536.66,575845.15,160792.48,151955.26,490547.05,143577.32
3,2012-07-07,2667930.94,559684.319357,73,364,73,6,24648.68,9769.32,81388.95,2552123.99,81388.95,24648.68,1175598.9,1161535.47,186522.26,0.0,28467.36,9769.32,122237.51,189416.69,579431.55,102617.21,222137.6,578121.31,174924.29,151130.02,303291.94,244622.82
4,2012-07-08,2360851.9,552229.105134,88,394,87,8,0.0,28513.13,171002.44,2161336.33,171002.44,0.0,640219.84,1101365.45,356141.22,0.0,63609.82,28513.13,254359.72,140368.75,389450.58,65543.14,35132.98,582705.53,156714.99,326917.19,295377.71,114281.31


In [220]:
# dữ lại phòng trường hợp cần cho phần 2.
data_frame_final.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,order_date,customer_id,zip_x,order_status,payment_method,device_type,order_source,installments,ship_date,delivery_date,shipping_fee,return_quantity,return_date,return_reason,rating,review_date,is_returned,promo_name,start_date,end_date,applicable_category,stackable_flag,promo_channel_all_channels,promo_channel_email,promo_channel_in_store,promo_channel_online,promo_channel_social_media,base_revenue,final_discount,actual_revenue,Date,product_name,size,price,cogs,category_Casual,category_GenZ,category_Outdoor,category_Streetwear,segment_Activewear,segment_All-weather,segment_Balanced,segment_Everyday,segment_Performance,segment_Premium,segment_Standard,segment_Trendy,color_black,color_blue,color_green,color_orange,color_pink,color_purple,color_red,color_silver,color_white,color_yellow,zip_y,city,signup_date,age_group,gender_Female,gender_Male,gender_Non-binary,acquisition_channel_direct,acquisition_channel_email_campaign,acquisition_channel_organic_search,acquisition_channel_paid_search,acquisition_channel_referral,acquisition_channel_social_media
0,1,2400,7,1138.22,0.0,NaN,NaN,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,3,2012-07-07,2012-07-11,1.37,0,NaT,NaN,5,2012-07-24,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7967.54,0.0,7967.54,2012-07-04,VietMotion YY-09,1,1109.261061,1053.798008,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,1109,39,2020-06-06,3,True,False,False,False,False,False,False,False,True
1,2,609,7,10166.25,0.0,NaN,NaN,2012-07-04,58621,1330,returned,cod,mobile,paid_search,1,2012-07-06,2012-07-10,2.60,6,2012-07-25,late_delivery,-1,NaT,1,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71163.75,0.0,71163.75,2012-07-04,SaigonFlex UC-74,2,10426.571034,8987.704231,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,1330,40,2021-11-03,1,True,False,False,False,False,False,False,False,True
2,3,396,3,11220.33,0.0,NaN,NaN,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,3,2012-07-04,2012-07-07,2.38,0,NaT,NaN,5,2012-08-03,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33660.99,0.0,33660.99,2012-07-04,SaigonFlex UM-01,1,11028.428695,10091.012256,False,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,1473,27,2020-09-18,3,True,False,False,True,False,False,False,False,False
3,4,635,5,10639.25,0.0,NaN,NaN,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,3,2012-07-05,2012-07-11,2.49,0,NaT,NaN,-1,NaT,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,53196.25,0.0,53196.25,2012-07-04,SaigonFlex UC-00,4,10745.220588,9205.430478,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,2360,32,2016-05-29,4,False,True,False,True,False,False,False,False,False
4,6,1935,1,1597.84,0.0,NaN,NaN,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,1,2012-07-09,2012-07-16,25.79,0,NaT,NaN,-1,NaT,0,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1597.84,0.0,1597.84,2012-07-06,UrbanVN RP-10,4,1609.911509,1048.696357,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,2886,30,2017-07-11,1,False,True,False,False,False,False,False,False,True


In [221]:
daily_timeseries.shape

(3833, 29)

### 3.5. lưu kết quả

In [234]:
output_dir = os.path.join('..', 'data', 'processed_data')
os.makedirs(output_dir, exist_ok=True)
file_name = 'daily_sales_data_final.csv'
file_path = os.path.join(output_dir, file_name)
daily_timeseries.to_csv(file_path, index=False, encoding='utf-8-sig')

print(f"Thư mục lưu trữ: {os.path.abspath(output_dir)}") # In ra đường dẫn tuyệt đối để bạn dễ tìm


Thư mục lưu trữ: C:\Users\skvat\PycharmProjects\Lanh-It-s-Du-Lieu\data\processed_data


### Q1 — Median inter-order gap

In [ ]:
# TODO: Tính median số ngày giữa 2 lần mua liên tiếp của khách có >1 đơn
# Hint: orders.sort_values → groupby customer_id → diff order_date → median


### Q2 — Gross margin theo segment

In [ ]:
# TODO: (price - cogs) / price, group by segment


### Q3 — Return reason phổ biến nhất — Streetwear

In [ ]:
# TODO: join returns <-> products, filter category=='Streetwear', value_counts return_reason


### Q4 — Traffic source có bounce_rate thấp nhất

In [ ]:
# TODO: group web_traffic by traffic_source → mean bounce_rate → idxmin


### Q5 — % order_items có promo_id

In [ ]:
# TODO: order_items.promo_id.notna().mean() * 100


### Q6 — Age group có số đơn trung bình cao nhất

In [ ]:
# TODO: join customers <-> orders, filter age_group not null, group age_group


### Q7 — Region doanh thu cao nhất

In [ ]:
# TODO: join orders <-> geography, gán revenue theo order_date, group region


### Q8 — Payment method phổ biến trong đơn cancelled

In [ ]:
# TODO: filter orders status=='cancelled' → value_counts payment_method


### Q9 — Size có return rate cao nhất

In [ ]:
# TODO: dùng compute_return_rate(returns, order_items, products, 'size')


### Q10 — Installment plan có payment_value trung bình cao nhất

In [ ]:
# TODO: group payments by installments → mean payment_value


## 4. Tổng hợp đáp án

In [ ]:
answers = {
    'Q1': None,  # 'A' / 'B' / 'C' / 'D'
    'Q2': None,
    'Q3': None,
    'Q4': None,
    'Q5': None,
    'Q6': None,
    'Q7': None,
    'Q8': None,
    'Q9': None,
    'Q10': None,
}

for q, a in answers.items():
    print(f'{q}: {a}')